<a href="https://colab.research.google.com/github/SiyuL2025/math_629_final/blob/main/Transformer_v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformer LOB — v4: CNN-Comparable Evaluation

**Design goals (matching CNN v16):**
- Data loaded from Google Drive CSV (same path as CNN — no pipeline re-run needed)
- Backtest uses `future_return` column with non-overlapping `PREDICTION_HORIZON=10` stride
- Same fee: 1.5 bp
- Same threshold sweep: `np.linspace(0.35, 0.80, 19)`, Sharpe-maximizing
- Same early-stopping criterion: Directional F1 (macro F1 over Down+Up)
- Same GPU setup: `cudnn.benchmark`, `pin_memory`, `num_workers`

**Overfitting fixes vs. previous version:**
- Dropout 0.1 → 0.3 (major regularisation increase)
- Weight decay 1e-4 → 1e-2 (matches CNN)
- LR 3e-4 → 1e-4 (slower, less memorisation)
- `shuffle=True` for training loader (was False)
- DirF1 early stopping, patience 10 (was val-acc, patience 6)
- Max epochs 30 → 50 (early stopping handles convergence)

In [ ]:
import math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import gc

from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/629/'
df_full      = pd.read_csv(DATA_PATH + 'lob_full_v4.csv')
df_no_pca    = pd.read_csv(DATA_PATH + 'lob_no_pca_v4.csv')
df_no_market = pd.read_csv(DATA_PATH + 'lob_no_market_v4.csv')

META_COLS = ['label', 'future_return', 'split']
full_feat_cols      = [c for c in df_full.columns      if c not in META_COLS]
no_pca_feat_cols    = [c for c in df_no_pca.columns    if c not in META_COLS]
no_market_feat_cols = [c for c in df_no_market.columns if c not in META_COLS]

split_idx = int((df_full['split'] == 'train').sum())

print('lob_full      shape:', df_full.shape,      '| features:', len(full_feat_cols))
print('lob_no_pca    shape:', df_no_pca.shape,    '| features:', len(no_pca_feat_cols))
print('lob_no_market shape:', df_no_market.shape, '| features:', len(no_market_feat_cols))
print('Train rows:', split_idx, '| Test rows:', len(df_full) - split_idx)
gc.collect()

Mounted at /content/drive


In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
SEED               = 42
SEQ_LEN            = 30        # longer look-back than CNN (10) — Transformer advantage
D_MODEL            = 64
N_HEADS            = 4
N_LAYERS           = 3
D_FF               = 128
DROPOUT            = 0.3       # was 0.1 — major overfitting fix
BATCH_SIZE         = 256
EPOCHS             = 50        # early stopping handles convergence
LR                 = 1e-4      # was 3e-4 — slower, less memorisation
PATIENCE           = 10        # matches CNN
WD                 = 1e-2      # was 1e-4 — matches CNN weight decay
EMBARGO_GAP        = 10        # skip 10 rows at train/test boundary (matches CNN)
PREDICTION_HORIZON = 10        # 10-second non-overlapping stride (matches CNN)

# ── Trading parameters ───────────────────────────────────────────────────────
FEE = 0.00015   # 1.5 bps — same as CNN

# ── Device setup ─────────────────────────────────────────────────────────────
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected — running on CPU will be slow.')
    print('Go to Runtime → Change runtime type → Hardware accelerator → GPU (T4)')

print(f'Device : {DEVICE}')
print(f'Config  — D_MODEL={D_MODEL} | N_HEADS={N_HEADS} | N_LAYERS={N_LAYERS} | DROPOUT={DROPOUT} | WD={WD}')
print(f'Trading — Fee={FEE*10000:.1f} bps | EMBARGO_GAP={EMBARGO_GAP} | PREDICTION_HORIZON={PREDICTION_HORIZON}')


def select_features(feature_set):
    if feature_set == 'full':
        df, feat_cols = df_full, full_feat_cols
    elif feature_set == 'no_pca':
        df, feat_cols = df_no_pca, no_pca_feat_cols
    elif feature_set == 'no_market':
        df, feat_cols = df_no_market, no_market_feat_cols
    else:
        raise ValueError('Unknown feature_set: ' + feature_set)
    print('Feature set:', feature_set.upper(), '| features:', len(feat_cols))
    X = df[feat_cols].values.astype(np.float32)
    y = (df['label'].values + 1).astype(np.int64)   # {-1,0,1} → {0,1,2}
    print('X shape:', X.shape, '| y shape:', y.shape)
    print('Label dist (down/flat/up):',
          np.round(np.bincount(y, minlength=3) / len(y), 3))
    return X, y


class LOBDataset(Dataset):
    def __init__(self, X, y, seq_len):
        self.X       = torch.tensor(X, dtype=torch.float32)
        self.y       = torch.tensor(y, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.X) - self.seq_len + 1

    def __getitem__(self, i):
        return self.X[i : i + self.seq_len], self.y[i + self.seq_len - 1]


def build_loaders(X, y):
    """Embargo gap of 10 rows at train/test boundary — matches CNN."""
    _pin        = (DEVICE.type == 'cuda')
    local_split = min(split_idx, len(X) - 1)
    X_tr = X[:local_split];  X_te = X[local_split + EMBARGO_GAP:]
    y_tr = y[:local_split];  y_te = y[local_split + EMBARGO_GAP:]

    y_tr_targets = y_tr[SEQ_LEN - 1:]
    counts  = np.bincount(y_tr_targets, minlength=3).astype(float)
    class_w = torch.tensor(counts.sum() / (3 * counts),
                            dtype=torch.float32).to(DEVICE)

    train_loader = DataLoader(LOBDataset(X_tr, y_tr, SEQ_LEN),
                              batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=_pin)
    test_loader  = DataLoader(LOBDataset(X_te, y_te, SEQ_LEN),
                              batch_size=BATCH_SIZE * 2, shuffle=False,
                              num_workers=2, pin_memory=_pin)

    print('Train samples:', len(train_loader.dataset),
          '| Test samples:', len(test_loader.dataset))
    print('Class weights (down/flat/up):', class_w.cpu().numpy().round(3))
    return train_loader, test_loader, class_w

## Transformer Architecture

Pre-norm Transformer encoder with sinusoidal positional encoding and mean-pooling.
Overfitting is controlled by increasing dropout to 0.3, weight decay to 1e-2,
and halving the learning rate compared to the previous version.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class LOBTransformer(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.proj    = nn.Linear(n_features, D_MODEL)
        self.pos_enc = PositionalEncoding(D_MODEL, dropout=DROPOUT)
        enc_layer    = nn.TransformerEncoderLayer(
            D_MODEL, N_HEADS, D_FF, DROPOUT,
            activation='gelu', batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(
            enc_layer, N_LAYERS, norm=nn.LayerNorm(D_MODEL)
        )
        self.head = nn.Sequential(
            nn.Linear(D_MODEL, D_MODEL // 2), nn.GELU(),
            nn.Dropout(DROPOUT), nn.Linear(D_MODEL // 2, 3)
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.pos_enc(self.proj(x))   # (B, T, D_MODEL)
        x = self.encoder(x).mean(dim=1)  # mean-pool over time → (B, D_MODEL)
        return self.head(x)


def build_model(n_features):
    model    = LOBTransformer(n_features).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print('Model params: %s  |  Input features: %d' % (f'{n_params:,}', n_features))
    return model

## Training

Early stopping on **Directional F1** (macro F1 over Down+Up only, same as CNN v16).
Label smoothing 0.05 prevents overconfident softmax outputs.

In [ ]:
def train(model, train_loader, test_loader, class_w):
    criterion = nn.CrossEntropyLoss(weight=class_w, label_smoothing=0.05)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_dir_f1': []}
    best_dir_f1, best_state, patience_cnt = 0.0, None, 0

    for epoch in range(1, EPOCHS + 1):
        model.train(); running = 0.0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running += loss.item() * len(yb)
        scheduler.step()
        train_loss = running / len(train_loader.dataset)

        model.eval()
        vloss, correct, total = 0.0, 0, 0
        ep_preds, ep_true = [], []
        with torch.no_grad():
            for Xb, yb in test_loader:
                Xb, yb  = Xb.to(DEVICE), yb.to(DEVICE)
                logits   = model(Xb)
                vloss   += criterion(logits, yb).item() * len(yb)
                preds    = logits.argmax(1)
                correct += (preds == yb).sum().item()
                total   += len(yb)
                ep_preds.append(preds.cpu().numpy())
                ep_true.append(yb.cpu().numpy())

        val_loss = vloss / total
        val_acc  = correct / total
        all_preds = np.concatenate(ep_preds)
        all_true  = np.concatenate(ep_true)
        dir_f1 = f1_score(all_true, all_preds,
                          labels=[0, 2], average='macro', zero_division=0)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_dir_f1'].append(dir_f1)

        print('Ep %3d/%d | Train %.4f | Val %.4f | Acc %.4f | DirF1 %.4f'
              % (epoch, EPOCHS, train_loss, val_loss, val_acc, dir_f1))

        if dir_f1 > best_dir_f1:
            best_dir_f1, patience_cnt = dir_f1, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print('Early stop at epoch %d. Best DirF1: %.4f' % (epoch, best_dir_f1))
                break

    model.load_state_dict(best_state)
    return history, best_dir_f1


def evaluate(model, test_loader):
    """Returns predictions, true labels, and softmax probabilities."""
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for Xb, yb in test_loader:
            logits = model(Xb.to(DEVICE))
            probs  = torch.softmax(logits, dim=-1)
            all_preds.append(logits.argmax(1).cpu().numpy())
            all_probs.append(probs.cpu().numpy())
            all_labels.append(yb.numpy())
    y_pred = np.concatenate(all_preds)
    y_prob = np.concatenate(all_probs)
    y_true = np.concatenate(all_labels)
    print(classification_report(y_true, y_pred,
                                target_names=['Down', 'Flat', 'Up'],
                                zero_division=0))
    return y_pred, y_true, y_prob


def plot_training(history, y_pred, y_true, title=''):
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    fig.suptitle('Transformer LOB v4 -- ' + title, fontsize=14)
    ep = range(1, len(history['train_loss']) + 1)

    axes[0].plot(ep, history['train_loss'], label='Train', color='steelblue')
    axes[0].plot(ep, history['val_loss'],   label='Val',   color='tomato')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(ep, history['val_acc'], color='seagreen')
    axes[1].axhline(max(history['val_acc']), color='tomato', linestyle='--',
                    label='Best=%.4f' % max(history['val_acc']))
    axes[1].set_title('Val Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)

    axes[2].plot(ep, history['val_dir_f1'], color='darkorange')
    axes[2].axhline(max(history['val_dir_f1']), color='tomato', linestyle='--',
                    label='Best=%.4f' % max(history['val_dir_f1']))
    axes[2].set_title('Directional F1'); axes[2].legend(); axes[2].grid(alpha=0.3)

    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[3],
                xticklabels=['Down', 'Flat', 'Up'],
                yticklabels=['Down', 'Flat', 'Up'])
    axes[3].set_title('Confusion Matrix')
    axes[3].set_xlabel('Predicted'); axes[3].set_ylabel('True')
    plt.tight_layout(); plt.show()

## Backtest Engine + Threshold Sweep

**Identical to CNN v16:**
- Uses `future_return` (10-second forward return from the pipeline CSV)
- Strides predictions by `PREDICTION_HORIZON=10` — non-overlapping P&L windows
- Annualization: `365 × 24 × 3600 / 10` (24/7 crypto convention)
- Fee: 1.5 bp one-way
- Threshold sweep: `np.linspace(0.35, 0.80, 19)`

In [ ]:
def simple_backtest(y_pred_labels, future_returns, fee=FEE, exp_name='Strategy'):
    """
    Compounding backtest on NON-OVERLAPPING predictions.
    y_pred_labels : np.ndarray {0=Down, 1=Flat, 2=Up} — strided every PREDICTION_HORIZON
    future_returns: np.ndarray — future_return[t] = (price[t+H]-price[t])/price[t], H=10
    """
    signals           = y_pred_labels - 1
    trades            = np.abs(np.diff(signals, prepend=0))
    transaction_costs = trades * fee
    net_returns       = (signals * future_returns) - transaction_costs
    cum_returns       = np.cumprod(1 + net_returns)

    total_return  = cum_returns[-1] - 1
    annual_factor = 365 * 24 * 3600 / PREDICTION_HORIZON
    sharpe        = np.mean(net_returns) / (np.std(net_returns) + 1e-8) * np.sqrt(annual_factor)
    running_max   = np.maximum.accumulate(cum_returns)
    max_drawdown  = np.min((cum_returns - running_max) / running_max)
    active        = signals != 0
    win_rate      = (net_returns[active] > 0).sum() / (active.sum() + 1e-8)
    turnover      = int(trades.sum())

    print('\n' + '-' * 55)
    print(f'  BACKTEST: {exp_name}  (Fee: {fee*10000:.1f} bps)')
    print('-' * 55)
    print(f'  Total Net Return : {total_return:>10.2%}')
    print(f'  Annualized Sharpe: {sharpe:>10.2f}')
    print(f'  Max Drawdown     : {max_drawdown:>10.2%}')
    print(f'  Win Rate         : {win_rate:>10.2%}')
    print(f'  Active steps     : {active.sum():>10,} / {len(signals):,}  ({active.mean():.1%})')
    print(f'  Total Turnover   : {turnover:>10,}  (position changes)')

    return cum_returns, total_return, sharpe, max_drawdown


def sweep_thresholds(y_prob, future_returns, fee=FEE, exp_name=''):
    """
    Per-class threshold sweep on NON-OVERLAPPING predictions.
    Identical logic to CNN v16.
    """
    print('\n' + '=' * 60)
    print(f'THRESHOLD SWEEP — {exp_name}  (Fee: {fee*10000:.1f} bps)')
    print('=' * 60)

    thresholds    = np.linspace(0.35, 0.80, 19)
    results       = []
    annual_factor = 365 * 24 * 3600 / PREDICTION_HORIZON

    for th in thresholds:
        preds = np.ones(len(y_prob), dtype=np.int64)
        preds[y_prob[:, 0] > th] = 0
        preds[y_prob[:, 2] > th] = 2

        signals = preds - 1
        trades  = np.abs(np.diff(signals, prepend=0))
        net_ret = (signals * future_returns) - (trades * fee)
        cum_ret = np.cumprod(1 + net_ret)

        sharpe    = np.mean(net_ret) / (np.std(net_ret) + 1e-8) * np.sqrt(annual_factor)
        total_ret = cum_ret[-1] - 1
        results.append({
            'Threshold'    : round(float(th), 4),
            'Total Return' : total_ret,
            'Sharpe Ratio' : sharpe,
            'Total Trades' : int(trades.sum())
        })

    df_res   = pd.DataFrame(results)
    best_idx = df_res['Sharpe Ratio'].idxmax()
    best_row = df_res.loc[best_idx]

    print(f'Best threshold : {best_row["Threshold"]:.4f}')
    print(f'  Sharpe       : {best_row["Sharpe Ratio"]:.2f}')
    print(f'  Net Return   : {best_row["Total Return"]:.2%}')
    print(f'  Turnover     : {best_row["Total Trades"]:,}')

    fig, ax1 = plt.subplots(figsize=(10, 5))
    color1 = 'tab:red'
    ax1.set_xlabel('Confidence Threshold')
    ax1.set_ylabel('Annualized Sharpe Ratio', color=color1, fontweight='bold')
    l1 = ax1.plot(df_res['Threshold'], df_res['Sharpe Ratio'],
                  color=color1, marker='o', label='Sharpe Ratio')
    ax1.axvline(best_row['Threshold'], color=color1, linestyle=':', alpha=0.6,
                label=f'Best th={best_row["Threshold"]:.4f}')
    ax1.tick_params(axis='y', labelcolor=color1); ax1.grid(alpha=0.3)

    ax2 = ax1.twinx()
    color2 = 'tab:blue'
    ax2.set_ylabel('Total Trades (Turnover)', color=color2, fontweight='bold')
    l2 = ax2.plot(df_res['Threshold'], df_res['Total Trades'],
                  color=color2, marker='x', linestyle='--', label='Total Trades')
    ax2.tick_params(axis='y', labelcolor=color2)

    lines  = l1 + l2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper center')
    plt.title(f'Threshold Sweep — {exp_name} | Fee: {fee*10000:.1f} bps', fontweight='bold')
    fig.tight_layout(); plt.show()

    return df_res, float(best_row['Threshold'])


def get_test_returns(df, split_idx):
    """
    NON-OVERLAPPING future_return values aligned to test predictions,
    strided every PREDICTION_HORIZON steps. Identical to CNN v16.
    """
    returns_all = df['future_return'].values
    start_idx   = split_idx + EMBARGO_GAP + SEQ_LEN - 1
    raw         = returns_all[start_idx:]
    return raw[::PREDICTION_HORIZON]

## Run Experiments

In [ ]:
def run_transformer(feature_set):
    print('=' * 65)
    print(f'  Transformer LOB v4 -- feature_set = {feature_set}')
    print('=' * 65)

    df = {'full': df_full, 'no_pca': df_no_pca, 'no_market': df_no_market}[feature_set]

    X, y                       = select_features(feature_set)
    train_ld, test_ld, class_w = build_loaders(X, y)
    model                      = build_model(n_features=X.shape[1])

    print('\n--- Training ---')
    history, best_dir_f1 = train(model, train_ld, test_ld, class_w)

    print('\n--- Evaluation ---')
    y_pred, y_true, y_prob = evaluate(model, test_ld)
    plot_training(history, y_pred, y_true, title=f'feature_set={feature_set}')

    # ── Align returns (non-overlapping stride) ──────────────────────────────
    test_returns = get_test_returns(df, split_idx)

    y_pred_s = y_pred[::PREDICTION_HORIZON]
    y_prob_s = y_prob[::PREDICTION_HORIZON]
    n        = min(len(y_pred_s), len(test_returns))
    y_pred_n  = y_pred_s[:n]
    y_prob_n  = y_prob_s[:n]
    returns_n = test_returns[:n]

    print(f'\nNon-overlapping steps: {n}  '
          f'(from {len(y_pred)} total predictions, stride={PREDICTION_HORIZON})')
    market_cum = np.cumprod(1 + returns_n)

    # ── Argmax backtest ──────────────────────────────────────────────────────
    print('\n--- Backtest: Argmax (no filter) ---')
    cum_argmax, ret_argmax, sh_argmax, dd_argmax = simple_backtest(
        y_pred_n, returns_n, fee=FEE, exp_name=f'Argmax ({feature_set})')

    # ── Threshold sweep ──────────────────────────────────────────────────────
    df_sweep, best_th = sweep_thresholds(
        y_prob_n, returns_n, fee=FEE, exp_name=f'Transformer ({feature_set})')

    # ── Optimal threshold backtest ───────────────────────────────────────────
    print(f'\n--- Backtest: Optimal threshold = {best_th:.4f} ---')
    y_pred_opt = np.ones(n, dtype=np.int64)
    y_pred_opt[y_prob_n[:, 0] > best_th] = 0
    y_pred_opt[y_prob_n[:, 2] > best_th] = 2

    cum_opt, ret_opt, sh_opt, dd_opt = simple_backtest(
        y_pred_opt, returns_n, fee=FEE,
        exp_name=f'Opt-th={best_th:.4f} ({feature_set})')

    # ── Equity curve ────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(market_cum,  label='Buy & Hold',               color='gray',      alpha=0.5, linewidth=1.5)
    ax.plot(cum_argmax,  label='Argmax (no filter)',        color='tomato',    linewidth=1.8, linestyle='--')
    ax.plot(cum_opt,     label=f'Threshold={best_th:.4f}', color='steelblue', linewidth=2)
    ax.set_title(f'Equity Curve — Transformer v4 ({feature_set})  Fee: {FEE*10000:.1f} bps  [non-overlapping]',
                 fontweight='bold')
    ax.set_ylabel('Cumulative Wealth')
    ax.set_xlabel(f'Prediction Steps (every {PREDICTION_HORIZON}s)')
    ax.legend(loc='upper left'); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    return {
        'feature_set' : feature_set,
        'best_dir_f1' : round(best_dir_f1, 4),
        'acc'         : round((y_pred == y_true).mean(), 4),
        'n_steps'     : n,
        'argmax_ret'  : ret_argmax,
        'argmax_sh'   : sh_argmax,
        'opt_th'      : best_th,
        'opt_ret'     : ret_opt,
        'opt_sh'      : sh_opt,
        'opt_dd'      : dd_opt,
        'model'       : model,
        'history'     : history,
    }

### Experiment 1: Full (PCA + Market)

In [ ]:
res_full = run_transformer('full')

### Experiment 2: No PCA (Raw LOB + Market)

In [ ]:
res_nopca = run_transformer('no_pca')

### Experiment 3: No Market (PCA only)

In [ ]:
res_nomarket = run_transformer('no_market')

## Ablation Summary

In [ ]:
print('=' * 75)
print(f'ABLATION SUMMARY  —  Transformer v4  |  Fee: {FEE*10000:.1f} bps')
print(f'Architecture: D_MODEL={D_MODEL} | N_HEADS={N_HEADS} | N_LAYERS={N_LAYERS} | DROPOUT={DROPOUT} | WD={WD}')
print('=' * 75)
print(f'{"Feature Set":<12} | {"Acc":>6} | {"DirF1":>7} | '
      f'{"Argmax Ret":>11} | {"Argmax Sh":>10} | '
      f'{"Opt-Th":>7} | {"Opt Ret":>9} | {"Opt Sh":>8} | {"Opt DD":>8}')
print('-' * 95)
for res in [res_full, res_nopca, res_nomarket]:
    print(f'{res["feature_set"]:<12} | '
          f'{res["acc"]:>6.4f} | '
          f'{res["best_dir_f1"]:>7.4f} | '
          f'{res["argmax_ret"]:>11.2%} | '
          f'{res["argmax_sh"]:>10.2f} | '
          f'{res["opt_th"]:>7.3f} | '
          f'{res["opt_ret"]:>9.2%} | '
          f'{res["opt_sh"]:>8.2f} | '
          f'{res["opt_dd"]:>8.2%}')